In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())
print("Source directory:", PROJECT_ROOT / "src")

Working directory: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026
Source directory: /Users/camillecu/Downloads/KUL/llm_project/llm_project2026/src


In [3]:
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env")

print("PG_CONN_STR configured:", bool(os.getenv("PG_CONN_STR")))

PG_CONN_STR configured: True


In [4]:
from index import text_search, vector_search, hybrid_search

print("Retrieval functions imported successfully.")

Retrieval functions imported successfully.


In [5]:
import psycopg

with psycopg.connect(os.environ["PG_CONN_STR"]) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                doc_id,
                symbol,
                year,
                quarter,
                COUNT(*) AS chunk_count
            FROM earnings_chunks
            GROUP BY doc_id, symbol, year, quarter
            ORDER BY symbol, year DESC, quarter DESC
        """)

        transcript_rows = cur.fetchall()

for row in transcript_rows:
    print(row)

('AAPL_2026Q3', 'AAPL', 2026, 3, 59)
('GOOG_2026Q2', 'GOOG', 2026, 2, 63)
('INTC_2026Q2', 'INTC', 2026, 2, 57)
('LRCX_2026Q4', 'LRCX', 2026, 4, 82)
('META_2026Q2', 'META', 2026, 2, 71)
('MSFT_2026Q4', 'MSFT', 2026, 4, 68)
('TSLA_2026Q2', 'TSLA', 2026, 2, 60)


In [6]:
# test retrieval manually
query = "What did Lam Research say about demand and outlook?"

for method_name, search_function in {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}.items():
    print(f"\n=== {method_name.upper()} ===")

    results = search_function(query, limit=5)

    for rank, result in enumerate(results, start=1):
        print(
            rank,
            result["doc_id"],
            result["symbol"],
            result["year"],
            result["quarter"],
            result.get("score", result.get("rrf_score")),
        )


=== TEXT ===
1 INTC_2026Q2 INTC 2026 2 0.14353834
2 INTC_2026Q2 INTC 2026 2 0.14353834
3 MSFT_2026Q4 MSFT 2026 4 0.1367836
4 INTC_2026Q2 INTC 2026 2 0.1367836
5 INTC_2026Q2 INTC 2026 2 0.1367836

=== VECTOR ===
1 GOOG_2026Q2 GOOG 2026 2 0.42016929863874397
2 GOOG_2026Q2 GOOG 2026 2 0.3997963250047587
3 GOOG_2026Q2 GOOG 2026 2 0.3263191878795647
4 GOOG_2026Q2 GOOG 2026 2 0.3203325783809926
5 GOOG_2026Q2 GOOG 2026 2 0.3146410209368442

=== HYBRID ===
1 INTC_2026Q2 INTC 2026 2 0.14353834
2 GOOG_2026Q2 GOOG 2026 2 0.42016929863874397
3 INTC_2026Q2 INTC 2026 2 0.14353834
4 GOOG_2026Q2 GOOG 2026 2 0.3997963250047587
5 MSFT_2026Q4 MSFT 2026 4 0.1367836


In [3]:
import sys
print(sys.version)
print(sys.executable)

3.11.5 (main, Sep 11 2023, 08:19:27) [Clang 14.0.6 ]
/Users/camillecu/Downloads/KUL/llm_project/llm_project2026/llmproject311/bin/python


# full comparison

In [5]:
from evaluate_retrieval import (
    evaluate_search_function,
    load_ground_truth,
    print_failures,
)

ground_truth = load_ground_truth(
    "data/retrieval_ground_truth.csv"
)

ground_truth_retrieval = [
    record
    for record in ground_truth
    if "NEEDS_CONTEXT" not in record["expected_chunk_ids"]
]

print("Retrieval-evaluation rows:", len(ground_truth_retrieval))

Retrieval-evaluation rows: 30


In [6]:
from index import text_search, vector_search, hybrid_search

search_methods = {
    "text": text_search,
    "vector": vector_search,
    "hybrid": hybrid_search,
}

evaluations = {}

for method_name, search_function in search_methods.items():
    evaluation = evaluate_search_function(
        search_function=search_function,
        ground_truth=ground_truth_retrieval,
        k=5,
    )

    evaluations[method_name] = evaluation

    print(f"\n=== {method_name.upper()} ===")
    print(f"Document Hit Rate@5: {evaluation['doc_hit_rate']:.3f}")
    print(f"Exact Chunk Hit Rate@5: {evaluation['chunk_hit_rate']:.3f}")
    print(f"Document MRR@5: {evaluation['doc_mrr']:.3f}")
    print(f"Exact Chunk MRR@5: {evaluation['chunk_mrr']:.3f}")


=== TEXT ===
Document Hit Rate@5: 0.833
Exact Chunk Hit Rate@5: 0.567
Document MRR@5: 0.789
Exact Chunk MRR@5: 0.483

=== VECTOR ===
Document Hit Rate@5: 0.700
Exact Chunk Hit Rate@5: 0.300
Document MRR@5: 0.667
Exact Chunk MRR@5: 0.267

=== HYBRID ===
Document Hit Rate@5: 0.833
Exact Chunk Hit Rate@5: 0.567
Document MRR@5: 0.740
Exact Chunk MRR@5: 0.418
